## Dataset overview
- users.csv: Customer information
- orders.csv: Transaction history
- subscriptions.csv: Subscription data
- preferences.csv: Customer taste preferences
- events.scv: User behavioral events (page views, cart actions)
- voucher_application: Voucher usage data
- user_references.csv: Referral tracking
- products.csv: Product catalog

## Key Question
- Understand what are different groups among current HK customer base
- How do they differ? What are their needs? 
- What are the implications for the marketing strategy in terms of attracting similar new customers and/or retaining current customers?
- How can these insights be used to recommend QLab Tea’s next action for user growth in HK?


In [ ]:
FILTER_WARNINGS = False

In [ ]:
import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.DEBUG)

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


sns.set_style("darkgrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

if FILTER_WARNINGS:
    import warnings
    warnings.filterwarnings("ignore")

plt.set_loglevel('warning')


logging.info("Setup completed successfully")
logging.debug("Using ENV variables:")
logging.debug(f"FILTER_WARNINGS: {FILTER_WARNINGS}")
logging.warning("This is a warning message.")
logging.error("This is an error message.")
logging.critical("This is a critical message.")

In [ ]:
logging.debug("Loading datasets")

try:
    users_df = pd.read_csv("./original_data/users.csv")
    orders_df = pd.read_csv("./original_data/orders.csv")
    preferences_df = pd.read_csv("./original_data/preferences.csv")

    logging.info(f"Users: {len(users_df):,} rows")
    logging.info(f"Orders: {len(orders_df):,} rows")
except FileNotFoundError as e:
    logging.error(f"Error csv not found: {e}")
    raise
except Exception as e:
    logging.critical(f"Load csv error: {e}")
    raise

## Data Cleaning

In [ ]:
missing_values = {
    "Users": users_df.isnull().sum(),
    "Orders": orders_df.isnull().sum(),
}

for k, v in missing_values.items():
    logging.debug(f"{k}")
    logging.debug(f"{v}\n")

### Clean Orders (Keeping only HKD orders)

In [ ]:
conversion_rate = {
    "SGD": 1.0,
    "HKD": 0.17,
    "MYR": 0.31,
}


orders_clean_df = orders_df.copy()
logging.debug(f"Original rows: {len(orders_clean_df)}")
# Only keep completed orders
orders_clean_df = orders_clean_df[orders_clean_df["status"].isin(["Shipped", "Complete"])]
logging.debug(f"After filtering for completed orders: {len(orders_clean_df)}")
# Remove <=0 order totals
orders_clean_df = orders_clean_df[orders_clean_df["total_incl_tax"]>0]
logging.debug(f"Remove potentially order_totals<=0: {len(orders_clean_df)}")
# Remove all users that are not from HK
orders_clean_df = orders_clean_df[orders_clean_df["currency"]=="HKD"]
logging.debug(f"Filter currency HK: {len(orders_clean_df)}")

# Fix dates
# Remove the microseconds part to standardize the format (irrelevant to our analysis)
orders_clean_df["date_placed"] = orders_clean_df["date_placed"].str.extract(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')

# Convert to proper datetime
orders_clean_df["date_placed"] = pd.to_datetime(orders_clean_df["date_placed"], errors='coerce')

# Convert to epoch time (seconds since 1970-01-01)
orders_clean_df["date_placed_epoch"] = orders_clean_df["date_placed"].astype('int64') // 10**9
logging.debug(f"After fixing datetime: {len(orders_clean_df)}")

orders_clean_df = orders_clean_df.dropna(subset=["date_placed", "user_id"])
logging.debug(f"Drop na: {len(orders_clean_df)}")

logging.debug(f"Cleaned orders: {len(orders_clean_df):,} rows")
logging.debug(f"Date range: {orders_clean_df['date_placed'].min()} - {orders_clean_df['date_placed'].max()}")
logging.debug(f"Total revenue: {orders_clean_df['total_incl_tax'].sum():.3f}")

def parse_order_items(order_items_json):
    try:
        items = json.loads(order_items_json)
        num_items = sum(int(item.get("quantity", 1)) for item in items)
        product_ids = [item.get("product_id") for item in items]
        return pd.Series({
            "num_items": num_items,
            "product_ids": product_ids
        })
    except:
        return pd.Series({"num_items": 1, "product_ids": []})
    
order_items_parsed = orders_clean_df['order_items'].apply(parse_order_items)
orders_clean_df = pd.concat([orders_clean_df, order_items_parsed], axis=1)

orders_clean_df["total_incl_tax_sgd"] = (
    orders_clean_df["total_incl_tax"]
    * orders_clean_df["currency"].map(conversion_rate)
)

logging.debug("Order items parsing done")
logging.info(f"Average item per order: {orders_clean_df['num_items'].mean():.2f}")
logging.debug(f"Final number of rows: {len(orders_clean_df)}")
logging.debug(f"Unique user ids: {orders_clean_df['user_id'].nunique()}")
display(orders_clean_df.head())

del orders_df

### Note
- Why are there only around 40 HK customers?
- Can we even do data analysis on a dataset of this size? Will the result even be reliable?
- We can attempt a simple RFM model

### Clean Users (Keeping only HKD users)

In [ ]:
users_clean_df = users_df.copy()

users_clean_df = users_clean_df[users_clean_df["id"].isin(orders_clean_df["user_id"])]
logging.debug(f"Number of users: {len(users_clean_df)}")
display(users_clean_df.head())

del users_df

## Feature Engineering

#### Time-Dependent RFM definition
- recency_days = (reference_date - max(order_date))
- frequency_orders = count of orders of a customer / (max(order_date)-min(order_date))
- monetary_total = weighted sum of total_incl_tax_sgd (weight is based on year, 2020: 0.2, 2021: 0.5, 2022: 0.8, 2023: 1.0)

In [ ]:
newest_date = orders_clean_df["date_placed"].max()
oldest_date = orders_clean_df["date_placed"].min()

logging.debug(f"Reference date: {newest_date}")

def arbitrary_date_weights(dates):
    year = dates.dt.year
    return np.select(
        [year <= 2020, year <= 2021, year <= 2022],
        [0.2, 0.5, 0.8],
        default=1.0,
    )

orders_clean_df["monetary_weight"] = arbitrary_date_weights(orders_clean_df["date_placed"])

orders_clean_df["weighted_total_incl_tax_sgd"] = (
    orders_clean_df["total_incl_tax_sgd"] * orders_clean_df["monetary_weight"]
)

rfm_features = orders_clean_df.groupby("user_id").agg(
    {
        "date_placed": [
            lambda x: (newest_date - x.max()).days,                     # Recency
            lambda x: len(x) / max((x.max() - x.min()).days, 1)         # Frequency
        ],
        "weighted_total_incl_tax_sgd": "sum" # Monetary
    }
).reset_index()

rfm_features.columns = ["user_id", "recency_days", "frequency_orders", "monetary_total"]

order_counts = orders_clean_df.groupby("user_id")["id"].count().rename("order_count")
rfm_features = rfm_features.join(order_counts, on="user_id")


logging.debug(f"\n{rfm_features.describe()}")
display(rfm_features.tail())

## Interpretation

### RFM model
1. Ternary model
2. Binary model

In [ ]:
def rank_based_rfm(df):
    
    df['R_Score'] = pd.qcut(df['recency_days'], q=3, labels=[3, 2, 1], duplicates='drop')
    # Use rank-based approach to handle frequency
    df['F_Score'] = pd.qcut(df['frequency_orders'].rank(method='first'), q=3, labels=[1, 2, 3])
    df['M_Score'] = pd.qcut(df['monetary_total'], q=3, labels=[1, 2, 3], duplicates='drop')
    
    # Create RFM score
    df['RFM_Score'] = df['R_Score'].astype(str) + df['F_Score'].astype(str) + df['M_Score'].astype(str)
    
    segment_map = {
        # R=3 (Most Recent), F=3 (Most Frequent), M=3 (Highest Value) - best
        '333': 'Champions',
        '332': 'Loyal Champions', 
        '331': 'Regular Champions',
        '323': 'Recent Loyal',
        '322': 'Core Loyal',
        '321': 'Developing Loyal',
        '313': 'Recent Big Spenders',
        '312': 'Recent Medium Spenders', 
        '311': 'Recent Customers',
        
        # R=2 (Medium Recency) - need attention
        '233': 'At-Risk Champions',
        '232': 'At-Risk Loyal',
        '231': 'At-Risk Regulars',
        '223': 'Slipping Big Spenders',
        '222': 'Average Customers',
        '221': 'Slipping Low-Value',
        '213': 'Occasional Big Spenders',
        '212': 'Occasional Medium Spenders',
        '211': 'Occasional Low-Spenders',
        
        # R=1 (Least Recent) - worst
        '133': 'Dormant Big Spenders',
        '132': 'Dormant Medium Spenders',
        '131': 'Dormant Customers',
        '123': 'Inactive Loyal',
        '122': 'Inactive Regulars',
        '121': 'Inactive Trialists',
        '113': 'Lost Big Spenders',
        '112': 'Lost Medium Spenders',
        '111': 'Lost Customers'
    }
    
    df['Segment'] = df['RFM_Score'].map(segment_map).fillna('Others')
    return df

# Apply
df = rank_based_rfm(rfm_features)
logging.info(f"\n{df['Segment'].value_counts()}")
display(df[["user_id", "Segment", "RFM_Score"]].head(15))

In [ ]:
RECENCY_THRESHOLD = 200 # around half a year

def binary_rfm(df):
    """
    Binary RFM scoring where:
    1 = Good, 0 = Bad
    Higher score = Better customer
    """
    # Calculate medians for binary splits
    recency_median = df['recency_days'].median()
    frequency_median = df['frequency_orders'].median()
    monetary_median = df['monetary_total'].median()
    
    logging.debug(f"Medians - Recency: {recency_median:.1f} days, Frequency: {frequency_median:.3f}, Monetary: ${monetary_median:.2f}")
    
    # Binary scoring (1 = Good, 0 = Bad)
    df['R_Score_binary'] = (df['recency_days'] < RECENCY_THRESHOLD).astype(int)  # 1 = Recent, 0 = Not recent
    df['F_Score_binary'] = (df['frequency_orders'] > frequency_median).astype(int)  # 1 = Frequent, 0 = Infrequent
    df['M_Score_binary'] = (df['monetary_total'] > monetary_median).astype(int)  # 1 = High value, 0 = Low value
    
    # Create binary RFM score
    df['RFM_Score_binary'] = df['R_Score_binary'].astype(str) + df['F_Score_binary'].astype(str) + df['M_Score_binary'].astype(str)
    
    # Binary segment map
    segment_map_binary = {
        '111': 'VIP Elite - Retain and Grow',
        '110': 'Loyal Core - Increase Value', 
        '101': 'High-Potential New - Build Frequency',
        '100': 'Emerging Customers - Nurture and Convert',
        '011': 'At-Risk VIP - Win-Back Priority',
        '010': 'Loyal but Declining - Re-engagement',
        '001': 'High-Value Inactive - Targeted Win-Back',
        '000': 'Dormant Base - Minimal Effort'
    }
    
    df['Segment_binary'] = df['RFM_Score_binary'].map(segment_map_binary)
    
    return df

df_binary = binary_rfm(df.copy())
logging.info(f"\n{df_binary['Segment_binary'].value_counts().sort_index()}")

## Note
- A simple interpretation may still be worth it for HK market
- Maybe include some basic preference data of these users and see if we can get a suggestion for what type of tea to push for